In [201]:
import math
%matplotlib inline
import torch
import random

class Value:
    def __init__(self, data=0.0, label='optional', children=()):
        self.data = data
        self.label = label
        self._backward = lambda: None
        self.grad = 0
        self._children = set(children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        result = Value(self.data * other.data, label = '*', children=(self, other))

        def _backward():
            self.grad += other.data * result.grad
            other.grad += self.data * result.grad

        result._backward = _backward
        return result

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        result = Value(self.data + other.data, label = '+', children=(self, other))

        def _backward():
            self.grad += result.grad
            other.grad += result.grad

        result._backward = _backward
        return result

    def tanh(self):
        result = Value(math.tanh(self.data), label = 'tanh', children=(self, ))

        def _backward():
            self.grad += (1 - result.data**2) * result.grad

        result._backward = _backward
        return result

    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children: 
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        self.grad = 1.0 # I think this should be 1, Carpathy used grad as 1. Make sense for multiplication DC/DC = 1
        for node in reversed(topo):
            #here I finally trigger the _backward function I was passing in every single __method__
            node._backward()

    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __neg__(self): return self * -1
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        # Explicitly name your arguments to avoid swaps!
        out = Value(self.data**other, children=(self,), label=f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out


In [202]:
x1 = torch.Tensor([2.00]).double()
x1.requires_grad = True
x2 = torch.Tensor([0.00]).double()
x2.requires_grad = True
w1 = torch.Tensor([-3.00]).double()
w1.requires_grad = True
w2 = torch.Tensor([1.00]).double()
w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()
b.requires_grad = True

n = x1*w1 + x2*w2 + b

o = torch.tanh(n)

o.backward()

print("o: ", o.data.item())
print("x1 grad: ", x1.grad.item())
print("x2 grad: ", x2.grad.item())
print("w1 grad: ", w1.grad.item())
print("w2 grad: ", w2.grad.item())

o:  0.7071066904050358
x1 grad:  -1.5000003851533106
x2 grad:  0.5000001283844369
w1 grad:  1.0000002567688737
w2 grad:  0.0


In [203]:
class Neuron:    
    def __init__(self, number_of_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range (number_of_inputs)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        result = sum((xi * wi for xi, wi in zip(x, self.w)), self.b) # sum can take second parameter to add to the sum like here with self.b

        return result.tanh()

    def parameters(self):
        return self.w + [self.b]
        
class Layer:
    def __init__(self, number_of_inputs, number_of_neurons):
        self.neurons = [Neuron(number_of_inputs) for _ in range(number_of_neurons)]

    def __call__(self, x):
        result = [neuron(x) for neuron in self.neurons]
        return result[0] if len(result) == 1 else result

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

class MLP:
    # MLP stands for Multi Layer Perceptron
    def __init__(self, number_of_inputs, list_of_numbers_of_neurons):
        layers = [number_of_inputs] + list_of_numbers_of_neurons
        self.layers = [Layer(layers[i], layers[i+1]) for i in range (len(list_of_numbers_of_neurons))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

        
x = [2.0, 3.0]
n = Neuron(2)
print("Neuron n: ", n(x))
l = Layer(2, 3)
print("Layer l: ", l(x))

x = [2.0, 3.0, -1.0]
mlp = MLP(3, [4, 4, 1])
print("MLP: ", mlp(x))


Neuron n:  Value(data=-0.8571600302411628, grad=0)
Layer l:  [Value(data=0.9982227304108625, grad=0), Value(data=0.9958062969989023, grad=0), Value(data=0.8559959492178982, grad=0)]
MLP:  Value(data=-0.06975472743254899, grad=0)


In [204]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] #desired targets

In [217]:
ypredictions = [mlp(x) for x in xs]
loss = sum([(y_output - y_ground_truth)**2 for y_ground_truth, y_output in zip(ys, ypredictions)])

for p in mlp.parameters():
    p.grad = 0.0

loss.backward()
mlp.parameters()
for p in mlp.parameters():
    p.data += -0.1 * p.grad

In [220]:
print ("ys: ", ys)
ypredictions


ys:  [1.0, -1.0, -1.0, 1.0]


[Value(data=0.8436751461678854, grad=-0.31264970766422917),
 Value(data=-0.9391362954268209, grad=0.12172740914635827),
 Value(data=-0.8059816699159289, grad=0.3880366601681422),
 Value(data=0.8428411325390504, grad=-0.3143177349218993)]

In [219]:
ypredictions

[Value(data=0.8436751461678854, grad=-0.31264970766422917),
 Value(data=-0.9391362954268209, grad=0.12172740914635827),
 Value(data=-0.8059816699159289, grad=0.3880366601681422),
 Value(data=0.8428411325390504, grad=-0.3143177349218993)]

In [222]:
print("kas vyksta?")
mlp.parameters()

kas vyksta?


[Value(data=0.9997189625680489, grad=0.008205794697000265),
 Value(data=-0.2403257691782148, grad=0.009412184060531298),
 Value(data=0.3745300705074404, grad=-0.008716336171201343),
 Value(data=0.6217797597035423, grad=0.005266852550495042),
 Value(data=-0.5366278552811482, grad=0.009000716044748749),
 Value(data=0.274980976214117, grad=0.02551538754624132),
 Value(data=-1.485959302057736, grad=0.06427317653743508),
 Value(data=0.36245990435722264, grad=0.03915764838254329),
 Value(data=1.1221088369062948, grad=-0.009510967384175843),
 Value(data=-0.7473398253955279, grad=-0.026378251686132736),
 Value(data=-0.15129126578928487, grad=-0.04026819173663045),
 Value(data=-0.25430333291059326, grad=-0.0255492639953217),
 Value(data=0.7790931623205583, grad=0.0018095135255992165),
 Value(data=0.31143265325752156, grad=0.004965716371626169),
 Value(data=0.08907184254790813, grad=0.007844944185085639),
 Value(data=0.6890522682841707, grad=0.00508172822306294),
 Value(data=-0.05766155583129744

In [230]:
def train_neural_network(neural_network, x_target, y_target, iteration_count, step_size):
    for iteration in range (iteration_count):
        # forward
        ypredictions = [neural_network(x) for x in x_target]
        loss = sum([(y_output - y_ground_truth)**2 for y_ground_truth, y_output in zip(y_target, ypredictions)])

        #go backward
        for p in neural_network.parameters():
            p.grad = 0.0
        loss.backward()

        #update
        for p in neural_network.parameters():
            p.data += -1 * step_size * p.grad

        print("Iteration: ", iteration, " loss data: ", loss.data, " y target: ", y_target, " y pred: ", ypredictions)


In [231]:
train_neural_network(mlp, xs, ys, 10, 0.01)

Iteration:  0  loss data:  0.0006154159757907884  y target:  [1.0, -1.0, -1.0, 1.0]  y pred:  [Value(data=0.984556796068708, grad=-0.030886407862583898), Value(data=-0.9929540841862119, grad=0.014091831627576212), Value(data=-0.988998925289881, grad=0.022002149420238037), Value(data=0.9856384244006896, grad=-0.02872315119862079)]
Iteration:  1  loss data:  0.0006152628112246913  y target:  [1.0, -1.0, -1.0, 1.0]  y pred:  [Value(data=0.984559279950589, grad=-0.03088144009882199), Value(data=-0.9929543683935607, grad=0.01409126321287868), Value(data=-0.988999277440414, grad=0.022001445119171903), Value(data=0.9856406771046206, grad=-0.02871864579075889)]
Iteration:  2  loss data:  0.0006151097375164586  y target:  [1.0, -1.0, -1.0, 1.0]  y pred:  [Value(data=0.9845617625644312, grad=-0.03087647487113765), Value(data=-0.9929546526354387, grad=0.01409069472912261), Value(data=-0.9889996297201818, grad=0.0220007405596363), Value(data=0.9856429286893523, grad=-0.028714142621295347)]
Iterati